# Segment QAT: Policy-Constrained Quantization-Aware Training

Fine-tunes the Mamba encoder to be robust to segment-level mixed-precision
quantization by sampling contiguous segment policies during training.

## Pipeline
1. **Cell 1**: Environment Setup + Drive Mount
2. **Cell 2**: Prerequisites Check
3. **Cell 3**: Train Segment QAT (~30-60 min for 50 epochs on GPU)
4. **Cell 4**: Evaluate PTQ baseline vs QAT
5. **Cell 5**: Comparison Plot (outage curves)
6. **Cell 6**: Summary Table

### Mathematical Formulation
```
L(theta) = l(f_theta(X), X) + lambda * E_{pi ~ p(Pi_seg)} [ l(f_theta(X; Q_pi(W)), X) ]
```
- First term: FP anchor loss (keeps model accurate at full precision)
- Second term: expected quantized loss over segment policies
- Policies sampled via DP at random budget levels (budget_sweep mode)

## Cell 1: Environment Setup

In [1]:
import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/MambaCompression"
MAMBAIC_ROOT = os.path.join(PROJECT_ROOT, "MambaIC")

if not os.path.isdir(PROJECT_ROOT):
    from google.colab import drive
    drive.mount('/content/drive')

assert os.path.isdir(MAMBAIC_ROOT), f"MambaIC not found: {MAMBAIC_ROOT}"
os.chdir(MAMBAIC_ROOT)
print(f"Working directory: {os.getcwd()}")

# Run setup script if available
setup_path = os.path.join(PROJECT_ROOT, "setup_colab.py")
if os.path.isfile(setup_path):
    print("Running setup_colab.py ...")
    exec(open(setup_path).read())
else:
    !pip install -q einops scipy tqdm thop fvcore pybind11

!pip install -q seaborn compressai timm pulp 2>/dev/null | tail -1

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n=== Environment Ready ===")

Working directory: /content/drive/MyDrive/MambaCompression/MambaIC
Running setup_colab.py ...
=== 1. Core Dependencies ===
[  0.0s] pip install core deps...
[  9.6s] core deps done

=== 2. VMamba CUDA Kernel (ss2d) ===
Current GPU: Tesla T4 (sm_75)
Cache arch matches current GPU (sm_75) ✓
Cache found! Restoring 1 kernel files...
[ 12.0s] copying .so from cache...
  Restored: selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so -> /usr/local/lib/python3.12/dist-packages/selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so
[ 12.0s] .so copy done
[ 12.1s] importing selective_scan_cuda_oflex...
[ 12.1s] selective_scan_cuda_oflex imported OK (sm_75)
selective_scan_cuda_oflex imported OK (sm_75)
[ 12.1s] setup_colab.py done

=== Setup Complete ===
Project: /content/drive/MyDrive/MambaCompression

CUDA available: True
Device: Tesla T4
GPU Memory: 15.6 GB

=== Environment Ready ===


## Cell 2: Prerequisites Check

Verifies that the required files exist:
- Pretrained model checkpoint
- Training and test data
- Cached segment omegas (from segment_dp_policy.py)
- Cached kappa, zeta, perfect rates (from rpmpq_v2.py)

In [2]:
import os, sys

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

RESULTS_CSV = os.path.join(MAMBAIC_ROOT, "results", "csv")

required_files = {
    "Pretrained checkpoint": "saved_models/mamba_transnet_L2_dim512_baseline/best.pth",
    "Train data (outdoor)": "data/DATA_Htrainout.mat",
    "Test data (outdoor)": "data/DATA_Htestout.mat",
    "Segment DP omegas": os.path.join(RESULTS_CSV, "segment_dp_omegas.csv"),
    "Kappa CSV": os.path.join(RESULTS_CSV, "rpmpq_v2_step1_nmse_kappa.csv"),
    "Zeta CSV": os.path.join(RESULTS_CSV, "rpmpq_v2_zeta.csv"),
    "Perfect rates CSV": os.path.join(RESULTS_CSV, "rpmpq_v2_perfect_rates.csv"),
}

all_ok = True
for name, path in required_files.items():
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}")
    if not exists:
        all_ok = False
        # Check alternate kappa path
        if "kappa" in name.lower():
            alt = os.path.join(RESULTS_CSV, "rpmpq_v2_kappa.csv")
            if os.path.isfile(alt):
                print(f"         -> Alternate found: {os.path.basename(alt)}")
                all_ok = True  # still OK

if all_ok:
    print("\nAll prerequisites satisfied. Ready to train.")
else:
    print("\nSome prerequisites missing. Run the following first:")
    print("  1. rpmpq_v2.py --step collect  (perturbation + kappa + zeta)")
    print("  2. segment_dp_policy.py         (segment omegas)")

  [OK] Pretrained checkpoint
  [OK] Train data (outdoor)
  [OK] Test data (outdoor)
  [OK] Segment DP omegas
  [OK] Kappa CSV
  [OK] Zeta CSV
  [OK] Perfect rates CSV

All prerequisites satisfied. Ready to train.


## Cell 3: Train Segment QAT

Fine-tunes the encoder with policy-constrained QAT.

**Key parameters:**
- `--epochs 50`: QAT fine-tuning epochs
- `--lambda-q 0.5`: weight for quantized loss term
- `--n-policies 4`: policies sampled per mini-batch
- `--sampling-mode budget_sweep`: solve DP at random budget each step
- `--saving-lo 85 --saving-hi 95`: BOPs saving range for sampling

Estimated time: **30-60 min** on T4/A100 GPU for 50 epochs.

In [3]:
!python analysis/segment_qat.py \
    --epochs 50 \
    --lr 1e-4 \
    --lambda-q 2 \
    --n-policies 4 \
    --sampling-mode budget_sweep \
    --saving-lo 85.0 \
    --saving-hi 97.0 \
    --clip-norm 1.0 \
    --batch-size 512

  SEGMENT QAT: Policy-Constrained Quantization-Aware Training

[1] Loading infrastructure (block structure, kappa, omega)...
    FC blocks: 32, segments: 177

[2] Loading model and data...
  Device: CUDA
[INFO] Building: UE Encoder [mamba-L2] + BS Decoder [transnet-L2]
/content/drive/MyDrive/MambaCompression/MambaIC/models/VSS_module.py:56: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/drive/MyDrive/MambaCompression/MambaIC/models/VSS_module.py:64: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/content/drive/MyDrive/MambaCompression/MambaIC/models/VSS_module.py:217: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/conte

## Cell 4: Evaluate PTQ vs QAT

Loads the QAT-trained model and runs outage evaluation.
Also runs the PTQ baseline (original pretrained model) for comparison.

Output: `segment_qat_comparison.csv`, comparison plots.

In [4]:
!python analysis/segment_qat.py --eval-only --compare

  SEGMENT QAT: Policy-Constrained Quantization-Aware Training

[1] Loading infrastructure (block structure, kappa, omega)...
    FC blocks: 32, segments: 177

[2] Loading model and data...
  Device: CUDA
[INFO] Building: UE Encoder [mamba-L2] + BS Decoder [transnet-L2]
/content/drive/MyDrive/MambaCompression/MambaIC/models/VSS_module.py:56: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/drive/MyDrive/MambaCompression/MambaIC/models/VSS_module.py:64: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/content/drive/MyDrive/MambaCompression/MambaIC/models/VSS_module.py:217: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/conte

## Cell 5: Comparison Plot

Visualize PTQ vs QAT outage curves and NMSE.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, "results", "csv")
RESULTS_PLOT = os.path.join(MAMBAIC_ROOT, "results", "plots")

# Load results
combined_csv = os.path.join(RESULTS_CSV, "segment_qat_comparison.csv")
if not os.path.exists(combined_csv):
    print("Comparison CSV not found. Run Cell 4 first.")
else:
    df = pd.read_csv(combined_csv)
    
    gammas = sorted(df["gamma"].unique())
    
    fig, axes = plt.subplots(1, len(gammas), figsize=(6 * len(gammas), 5), sharey=True)
    if len(gammas) == 1:
        axes = [axes]
    
    colors = {"PTQ": "#1f77b4", "QAT": "#d62728"}
    markers = {"PTQ": "o", "QAT": "s"}
    
    for ax, gamma in zip(axes, gammas):
        for label in ["PTQ", "QAT"]:
            sub = df[(df["label"] == label) & (df["gamma"] == gamma)]
            sub = sub.sort_values("target_saving")
            ax.plot(
                sub["target_saving"], sub["outage"],
                color=colors[label], marker=markers[label],
                label=label, linewidth=2, markersize=4,
            )
        ax.set_title(f"gamma = {gamma}", fontsize=13)
        ax.set_xlabel("BOPs Saving (%)")
        ax.set_ylim(-0.02, 1.02)
        if ax == axes[0]:
            ax.set_ylabel("Outage Probability")
            ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
    
    fig.suptitle("Rate-Based Outage: PTQ vs Segment QAT", fontsize=14)
    fig.tight_layout()
    plt.show()
    
    # NMSE plot
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    g0 = gammas[0]
    for label in ["PTQ", "QAT"]:
        sub = df[(df["label"] == label) & (df["gamma"] == g0)]
        sub = sub.sort_values("target_saving")
        ax2.plot(
            sub["target_saving"], sub["nmse_db"],
            color=colors[label], marker=markers[label],
            label=label, linewidth=2, markersize=4,
        )
    ax2.set_xlabel("BOPs Saving (%)")
    ax2.set_ylabel("NMSE (dB)")
    ax2.set_title("NMSE: PTQ vs Segment QAT")
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    plt.show()

## Cell 6: Summary Table

Tabular comparison at key saving levels.

In [ ]:
import os, sys
import pandas as pd
import numpy as np

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
RESULTS_CSV = os.path.join(MAMBAIC_ROOT, "results", "csv")

combined_csv = os.path.join(RESULTS_CSV, "segment_qat_comparison.csv")
if not os.path.exists(combined_csv):
    print("Run Cell 4 first.")
else:
    df = pd.read_csv(combined_csv)
    
    key_savings = [87.5, 90.0, 92.5, 95.0]
    gammas = sorted(df["gamma"].unique())
    
    for gamma in gammas:
        print(f"\n{'=' * 70}")
        print(f"  gamma = {gamma}")
        print(f"{'=' * 70}")
        print(f"{'Saving':>8s}  {'PTQ Outage':>12s}  {'QAT Outage':>12s}  {'Delta':>10s}  {'PTQ NMSE':>10s}  {'QAT NMSE':>10s}")
        print("-" * 70)
        
        for sav in key_savings:
            row_ptq = df[
                (df["label"] == "PTQ")
                & (df["gamma"] == gamma)
                & (df["target_saving"].between(sav - 0.3, sav + 0.3))
            ]
            row_qat = df[
                (df["label"] == "QAT")
                & (df["gamma"] == gamma)
                & (df["target_saving"].between(sav - 0.3, sav + 0.3))
            ]
            
            if len(row_ptq) > 0 and len(row_qat) > 0:
                o_ptq = row_ptq["outage"].values[0]
                o_qat = row_qat["outage"].values[0]
                n_ptq = row_ptq["nmse_db"].values[0]
                n_qat = row_qat["nmse_db"].values[0]
                delta = o_ptq - o_qat
                print(f"{sav:8.1f}%  {o_ptq:12.4f}  {o_qat:12.4f}  {delta:+10.4f}  {n_ptq:10.2f}  {n_qat:10.2f}")
    
    # QAT training history
    history_csv = os.path.join(MAMBAIC_ROOT, "saved_models",
                               "mamba_transnet_L2_dim512_qat", "qat_history.csv")
    if os.path.exists(history_csv):
        hist = pd.read_csv(history_csv)
        print(f"\n\nQAT Training History:")
        print(f"  Best val NMSE: {hist['val_nmse_db'].min():.2f} dB (epoch {hist.loc[hist['val_nmse_db'].idxmin(), 'epoch']})")
        print(f"  Final loss: {hist['train_loss'].iloc[-1]:.5f}")
        print(f"  Final Lfp: {hist['train_loss_fp'].iloc[-1]:.5f}")
        print(f"  Final Lq:  {hist['train_loss_q'].iloc[-1]:.5f}")